In [ ]:
import requests
import json
import math
import pandas as pd
import numpy as np

def shot_distance(x, y):
    goal_x, goal_y = 120, 40
    return math.hypot(goal_x - x, goal_y - y)

def shot_angle(x, y):
    if x > 60:
        left_post = (120, 36)
        right_post = (120, 44)
    else:
        left_post = (0, 36)
        right_post = (0, 44)

    a = (left_post[0] - x, left_post[1] - y)
    b = (right_post[0] - x, right_post[1] - y)

    dot = a[0]*b[0] + a[1]*b[1]
    mag_a = math.hypot(*a)
    mag_b = math.hypot(*b)
    angle = math.acos(dot / (mag_a * mag_b))
    return math.degrees(angle)

def find_event_by_id(events, event_id):
    for e in events:
        if e["id"] == event_id:
            return e
    return None

season_ids = [90, 4, 1, 42]
train_shots = []
test_shots = []

for season_id in season_ids:

    url = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/11/{season_id}.json"

    response = requests.get(url)
    matches = response.json()
    sorted_matches = sorted(matches, key=lambda x: x["match_week"])

    match_ids = [match["match_id"] for match in sorted_matches]
    match_weeks = [match["match_week"] for match in sorted_matches]
    

    for match_id in match_ids:
        url_match = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/{match_id}.json"
        response_match = requests.get(url_match)
        events = response_match.json()


        for event in events:
            if event["type"]["name"] == "Shot" and event["team"]["name"] == "Barcelona":
                x_loc1, y_loc1 = event["location"]

                def euclid(a, b):
                    return math.hypot(a[0]-b[0], a[1]-b[1])

                def safe_get_gk(freeze_frame):
                    for p in freeze_frame:
                        pos = p.get("position", {}).get("name", "").lower()
                        if "goalkeeper" in pos:
                            return p
                    return None

                def extract_freezefeature(event):

                    ff = event.get("shot", {}).get("freeze_frame", None)
                    features = {
                        "n_def_1_5": 0,
                        "n_def_3_0": 0,
                        "dist_nearest_def": 99.0 / 127,
                        "gk_dist_to_shooter": 11 / 127,
                    }

                    sx, sy = event.get("location", [np.nan, np.nan])

                    if not ff:
                        return features

                    defender_distances = []
                    for p in ff:
                        if not p["teammate"]:
                            loc = p["location"]
                            d = euclid((sx, sy), (loc[0], loc[1]))
                            defender_distances.append(d)

                    if defender_distances:
                        features["n_def_1_5"] = sum(1 for d in defender_distances if d <= 1.5)
                        features["n_def_3_0"] = sum(1 for d in defender_distances if d <= 3.0)
                        features["dist_nearest_def"] = float(np.min(defender_distances)) / 127
                    else:
                        features["n_def_1_5"] = 0
                        features["n_def_3_0"] = 0
                        features["dist_nearest_def"] = 99.0 / 127

                    gkp = safe_get_gk(ff)
                    if gkp:
                        gx, gy = gkp.get("location", [np.nan, np.nan])
                        features["gk_dist_to_shooter"] = euclid((sx, sy), (gx, gy)) / 127
                    else:
                        features["gk_dist_to_shooter"] = 11 / 127

                    return features

                ff_feats = extract_freezefeature(event)

                shot = {
                    "week": next(m["match_week"] for m in sorted_matches if m["match_id"] == match_id),
                    "time": f"{str(event['minute']).zfill(2)}:{str(event['second']).zfill(2)}",
                    "player": event["player"]["name"],
                    "distance": round(shot_distance(x_loc1, y_loc1), 2),
                    "angle": round(shot_angle(x_loc1, y_loc1), 0),
                    "type": event["shot"]["type"]["name"],
                    "technique": event["shot"]["technique"]["name"],
                    "outcome": event["shot"]["outcome"]["name"],
                    "goal/no goal": 1 if event["shot"]["outcome"]["name"] == "Goal" else 0,
                    "type_b": 0 if event["shot"]["type"]["name"] == "Open Play" else 1,
                    "technique_b": 0 if event["shot"]["technique"]["name"] == "Normal" else 1,
                    "distance_d": round(shot_distance(x_loc1, y_loc1), 2) / 127,
                    "distance_l": math.log1p(round(shot_distance(x_loc1, y_loc1), 2)),
                    "angle_d": round(shot_angle(x_loc1, y_loc1), 0) / 180,
                    "free_kick_flag": 1 if event["shot"]["type"]["name"] == "Free Kick" else 0,
                    "penalty_flag": 1 if event["shot"]["type"]["name"] == "Penalty" else 0
                }
                shot.update(ff_feats)
                if season_id == 42:
                    shot["statsbomb_xg"] = event["shot"]["statsbomb_xg"]
                    test_shots.append(shot)
                else:
                    train_shots.append(shot)

print(f"Total Barcelona training shots collected: {len(train_shots)}")
print(f"Total Barcelona testing shots collected: {len(test_shots)}")
df_train = pd.DataFrame(train_shots)
df_test = pd.DataFrame(test_shots)

df_train.insert(0, "shot_id", range(1, len(df_train) + 1))
df_test.insert(0, "shot_id", range(1, len(df_test) + 1))

df_train.to_excel("training_barcelona_shots.xlsx", index=False)
df_test.to_excel("testing_barcelona_shots.xlsx", index=False)

print("Jaggernaut")

In [ ]:
# GET PASSES AND ITS DONE